# DINOv3-BiGRU-GatedABMIL Frozen Baseline on Vast.ai

This notebook prepares CQ500/Qure.ai HeadCT, extracts frozen DINOv3 slice features, and trains the scan-level BiGRU + Gated ABMIL baseline.

Expected environment variables in `.env` or `.env.txt`:

- `HF_KEY`
- `KAGGLE_USERNAME`
- `KAGGLE_KEY`

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

ROOT = Path.cwd()
print(ROOT)
assert (ROOT / 'requirements.txt').exists(), 'Run this notebook from the repo root.'

## 1. Install dependencies

Run this once on a fresh Vast.ai instance. Restart the kernel after installation if Jupyter asks.

In [ ]:
!python -m pip install --upgrade pip
!python -m pip install -r requirements.txt
!python -m pip install -e .

In [ ]:
import torch
print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

## 2. Download model and dataset

The Kaggle dataset is large. Make sure the instance has enough disk space.

In [ ]:
!python scripts/download_hf_model.py

In [ ]:
!python scripts/download_kaggle_dataset.py --unzip

## 3. Build labels, index, and split

Labels are majority vote over the three radiologist reads in `reads.csv`. Splits are case-level.

In [ ]:
!python scripts/normalize_labels.py --labels-csv data/raw/qureai-headct/reads.csv --id-column name
!python scripts/build_fast_path_index.py --dicom-root data/raw/qureai-headct --out-csv data/processed/dicom_index_fast.csv
!python scripts/filter_matched_labels.py
!python scripts/make_patient_split.py --labels-csv data/processed/labels_matched.csv --out-csv splits/cq500_seed42.csv

In [ ]:
import pandas as pd
labels = pd.read_csv('data/processed/labels_matched.csv')
splits = pd.read_csv('splits/cq500_seed42.csv')
index = pd.read_csv('data/processed/dicom_index_fast.csv', usecols=['case_id'])
print('labels:', labels.shape)
print('cases in index:', index.case_id.nunique())
print(splits.split.value_counts())
display(labels.drop(columns='case_id').sum().to_frame('positive_cases'))

## 4. Smoke test feature extraction

This verifies DICOM decoding and DINOv3 inference before launching the full extraction.

In [ ]:
!python scripts/extract_dinov3_features.py \
  --model data/raw/hf/dinov3-vitb16-pretrain-lvd1689m \
  --index-csv data/processed/dicom_index_fast.csv \
  --split-csv splits/cq500_seed42.csv \
  --split train \
  --limit-cases 2 \
  --max-slices 2 \
  --batch-size 1 \
  --out-dir data/features/dinov3_vitb16_smoke

## 5. Full DINOv3 feature extraction

Tune `BATCH_SIZE` for your GPU. If you hit out-of-memory, reduce it.

In [ ]:
BATCH_SIZE = 32
MODEL_DIR = 'data/raw/hf/dinov3-vitb16-pretrain-lvd1689m'
for split in ['train', 'val', 'test']:
    cmd = [
        sys.executable, 'scripts/extract_dinov3_features.py',
        '--model', MODEL_DIR,
        '--index-csv', 'data/processed/dicom_index_fast.csv',
        '--split-csv', 'splits/cq500_seed42.csv',
        '--split', split,
        '--batch-size', str(BATCH_SIZE),
        '--out-dir', 'data/features/dinov3_vitb16',
    ]
    print(' '.join(cmd))
    subprocess.run(cmd, check=True)

## 6. Train frozen baseline

In [ ]:
!python scripts/train_frozen_baseline.py --config configs/vast/frozen_baseline.yaml

In [ ]:
import json
metrics_path = Path('data/models/frozen_bigru_abmil/metrics.json')
if metrics_path.exists():
    print(json.dumps(json.loads(metrics_path.read_text()), indent=2))

## 7. Build Seg-CQ500 slice labels

Seg-CQ500 provides 3D hemorrhage segmentation masks for 51 CQ500 scans. We convert each mask volume into binary slice labels: a slice is positive when the corresponding mask slice contains at least one hemorrhage voxel.

In [ ]:
!python scripts/download_seg_cq500.py --unzip
!python scripts/build_seg_cq500_slice_labels.py \
  --seg-root data/raw/seg-cq500 \
  --index-csv data/processed/dicom_index_fast.csv \
  --out-csv data/processed/slice_labels.csv

In [ ]:
slice_labels = pd.read_csv('data/processed/slice_labels.csv')
print(slice_labels.shape)
display(slice_labels.head())
display(slice_labels.groupby('case_id')['hemorrhage'].agg(['count', 'sum']).head())
display(slice_labels['hemorrhage'].value_counts().to_frame('slices'))

## 8. Train frozen slice-level hemorrhage head

This is intentionally separate from the BiGRU-GatedABMIL scan classifier. It trains a small binary head on cached frozen DINOv3 slice embeddings to answer: is this slice a hemorrhage slice?

The previous cell writes `data/processed/slice_labels.csv` with `case_id`, `path`, `instance_number`, and `hemorrhage` columns.

In [ ]:
SLICE_LABELS = Path('data/processed/slice_labels.csv')
if not SLICE_LABELS.exists():
    print('Missing', SLICE_LABELS)
    print('Create it with columns: case_id, hemorrhage, path OR case_id, hemorrhage, instance_number')
else:
    slice_labels = pd.read_csv(SLICE_LABELS)
    print(slice_labels.shape)
    display(slice_labels.head())
    display(slice_labels['hemorrhage'].value_counts(dropna=False).to_frame('count'))

In [ ]:
# Edit configs/vast/slice_head.yaml if your slice label columns use different names.
if SLICE_LABELS.exists():
    !python scripts/train_slice_head.py --config configs/vast/slice_head.yaml

In [ ]:
slice_metrics_path = Path('data/models/slice_hemorrhage_head/metrics.json')
if slice_metrics_path.exists():
    print(json.dumps(json.loads(slice_metrics_path.read_text()), indent=2))

## Next step: LoRA model

After this frozen baseline is reproducible, add a second training path:

- DINOv3 + LoRA adapters
- scan-level BiGRU-GatedABMIL loss
- optional slice-level auxiliary head only for truly slice-labeled samples